# AUTO-LABELING: Belajar dari Contoh Manual
## Sentiment Analysis of Indonesian Political News

**Metodologi:** Semi-supervised annotation — code belajar pola labeling dari contoh manual,
lalu menerapkannya ke seluruh dataset.

**Input:**
- 2,400 artikel berlabel manual (seed)
- Dataset penuh yang belum berlabel

**Output:**
- Dataset final 25,322 artikel berlabel

**Referensi:** Zhu & Goldberg (2009). Introduction to Semi-Supervised Learning. *Morgan & Claypool.*

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

print('Libraries loaded!')

In [ ]:
# ================================================================
# PATHS
# ================================================================
MANUAL_PATH = '../../data/data_berita/train_data/'  # 2,400 artikel manual
RAW_PATH    = '../../data/data_berita/raw_fixed/'    # dataset mentah penuh
OUTPUT_PATH = '../../data/data_berita/cleaning/data_labeled/'
os.makedirs(OUTPUT_PATH, exist_ok=True)

In [ ]:
# ================================================================
# STEP 1: Load Data Manual (Contoh Labeling)
# ================================================================
df_manual_cnbc   = pd.read_csv(f'{MANUAL_PATH}/cnbc_train_data.csv')
df_manual_detik  = pd.read_csv(f'{MANUAL_PATH}/detik_train_data.csv')
df_manual_kompas = pd.read_csv(f'{MANUAL_PATH}/kompas_train_data.csv')

df_manual = pd.concat([df_manual_cnbc, df_manual_detik, df_manual_kompas], ignore_index=True)
df_manual = df_manual.dropna(subset=['text', 'label'])

print('=== DATA MANUAL (CONTOH LABELING) ===')
print(f'Total: {len(df_manual):,} artikel berlabel manual')
print(f'\nDistribusi label:')
vc = df_manual['label'].value_counts().sort_index()
for lbl, cnt in vc.items():
    name = {-1:'Negatif', 0:'Netral', 1:'Positif'}[lbl]
    print(f'  {name:10}: {cnt:,} ({cnt/len(df_manual)*100:.1f}%)')

In [ ]:
# ================================================================
# STEP 2: Load Dataset Penuh (Belum Berlabel)
# ================================================================
df_raw_cnbc   = pd.read_csv(f'{RAW_PATH}/cnbc_articles.csv')
df_raw_detik  = pd.read_csv(f'{RAW_PATH}/detik_articles.csv')
df_raw_kompas = pd.read_csv(f'{RAW_PATH}/kompas_articles.csv')

# Rename kolom agar konsisten
for df, src in [(df_raw_cnbc,'cnbc'),(df_raw_detik,'detik'),(df_raw_kompas,'kompas')]:
    if 'published_at' in df.columns:
        df.rename(columns={'published_at':'date'}, inplace=True)
    if 'url' in df.columns:
        df.drop(columns=['url'], inplace=True)
    df['text'] = (df['title'].fillna('') + '. ' + df['content'].fillna('')).str.strip()

df_raw_all = pd.concat([df_raw_cnbc, df_raw_detik, df_raw_kompas], ignore_index=True)
df_raw_all = df_raw_all.dropna(subset=['text'])
df_raw_all = df_raw_all[df_raw_all['text'].str.len() > 50]

print('=== DATASET PENUH ===')
print(f'Total: {len(df_raw_all):,} artikel')

# Identifikasi artikel yang sudah berlabel manual
manual_ids = set(df_manual['article_id'].tolist()) if 'article_id' in df_manual.columns else set()
manual_texts = set(df_manual['text'].str[:100].tolist())

# Artikel yang BELUM berlabel (akan di-auto-label)
df_unlabeled = df_raw_all[~df_raw_all['text'].str[:100].isin(manual_texts)].copy()
print(f'Sudah berlabel manual: {len(df_manual):,}')
print(f'Belum berlabel: {len(df_unlabeled):,}')

In [ ]:
# ================================================================
# STEP 3: Train Classifier dari Contoh Manual
# Code BELAJAR pola labeling dari 2,400 contoh manual
# ================================================================

# TF-IDF: ubah teks → vektor fitur
# Setiap kata jadi fitur, bobotnya berdasarkan frekuensi
print('Training classifier dari contoh manual...')

vectorizer = TfidfVectorizer(
    max_features=20000,      # pakai 20,000 kata paling penting
    ngram_range=(1, 2),      # unigram + bigram (contoh: 'korupsi', 'tidak korupsi')
    sublinear_tf=True,       # normalisasi frekuensi
    min_df=2                 # kata harus muncul minimal 2x
)

X = vectorizer.fit_transform(df_manual['text'].fillna(''))
y = df_manual['label'].values

# Class weights — beri perhatian lebih ke kelas minoritas
classes = np.unique(y)
class_weights = compute_class_weight('balanced', classes=classes, y=y)
weight_dict = dict(zip(classes, class_weights))

# Logistic Regression — belajar dari contoh manual
clf = LogisticRegression(
    class_weight=weight_dict,
    max_iter=1000,
    random_state=42,
    C=1.0
)
clf.fit(X, y)

# Evaluasi di data manual (internal check)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
clf_eval = LogisticRegression(class_weight=weight_dict, max_iter=1000, random_state=42)
clf_eval.fit(X_train, y_train)
y_pred_val = clf_eval.predict(X_val)

print('\n=== EVALUASI PADA DATA MANUAL (80/20 SPLIT) ===')
print(classification_report(y_val, y_pred_val,
      target_names=['Negatif','Netral','Positif'], digits=3))

In [ ]:
# ================================================================
# STEP 4: Auto-Label Sisa Artikel
# Classifier terapkan pola yang dipelajari ke artikel belum berlabel
# ================================================================

print(f'Auto-labeling {len(df_unlabeled):,} artikel...')

X_unlabeled = vectorizer.transform(df_unlabeled['text'].fillna(''))
df_unlabeled['label'] = clf.predict(X_unlabeled)

print('Distribusi label hasil auto-labeling:')
vc = df_unlabeled['label'].value_counts().sort_index()
for lbl, cnt in vc.items():
    name = {-1:'Negatif', 0:'Netral', 1:'Positif'}[lbl]
    print(f'  {name:10}: {cnt:,} ({cnt/len(df_unlabeled)*100:.1f}%)')

In [ ]:
# ================================================================
# STEP 5: Gabungkan Manual + Auto-Label → Dataset Final
# ================================================================

cols = ['date', 'title', 'content', 'article_id', 'text', 'label']

# Pastikan semua kolom ada
for c in ['article_id']:
    if c not in df_unlabeled.columns:
        df_unlabeled['article_id'] = [f'ART_{i:06d}' for i in range(len(df_unlabeled))]

# Gabung
df_final = pd.concat([
    df_manual[[c for c in cols if c in df_manual.columns]],
    df_unlabeled[[c for c in cols if c in df_unlabeled.columns]]
], ignore_index=True)

print('=== DATASET FINAL ===')
print(f'Manual label  : {len(df_manual):,}')
print(f'Auto label    : {len(df_unlabeled):,}')
print(f'TOTAL FINAL   : {len(df_final):,}')
print(f'\nDistribusi akhir:')
vc_final = df_final['label'].value_counts().sort_index()
for lbl, cnt in vc_final.items():
    name = {-1:'Negatif', 0:'Netral', 1:'Positif'}[lbl]
    print(f'  {name:10}: {cnt:,} ({cnt/len(df_final)*100:.1f}%)')

In [ ]:
# ================================================================
# STEP 6: Save per Portal
# ================================================================

# Split balik per portal berdasarkan article_id prefix
df_cnbc_out   = df_final[df_final['article_id'].str.startswith('CNB')]
df_detik_out  = df_final[df_final['article_id'].str.startswith('DET')]
df_kompas_out = df_final[df_final['article_id'].str.startswith('KOM')]

df_cnbc_out.to_csv(f'{OUTPUT_PATH}/cnbc_labeled.csv',   index=False, encoding='utf-8-sig')
df_detik_out.to_csv(f'{OUTPUT_PATH}/detik_labeled.csv',  index=False, encoding='utf-8-sig')
df_kompas_out.to_csv(f'{OUTPUT_PATH}/kompas_labeled.csv', index=False, encoding='utf-8-sig')

print('=== FILE TERSIMPAN ===')
print(f'cnbc_labeled.csv   → {len(df_cnbc_out):,} artikel')
print(f'detik_labeled.csv  → {len(df_detik_out):,} artikel')
print(f'kompas_labeled.csv → {len(df_kompas_out):,} artikel')
print(f'TOTAL              → {len(df_final):,} artikel')
print('\n✅ Selesai! Dataset siap digunakan untuk training model.')

## ✅ Pipeline Selesai!

### Cara Kerja:
1. **Load 2,400 contoh manual** — artikel yang sudah dilabeli tim
2. **TF-IDF vectorization** — ubah teks jadi angka (frekuensi kata)
3. **Logistic Regression** — belajar pola: kombinasi kata apa yang cenderung Positif/Negatif/Netral
4. **Prediksi** — terapkan pola yang dipelajari ke 22,900 artikel sisanya
5. **Gabungkan** — 2,400 manual + 22,900 auto = 25,322 artikel berlabel

### Kenapa Logistic Regression + TF-IDF?
- **Interpretable**: bisa lihat kata apa yang paling berpengaruh per kelas
- **Reproducible**: deterministik, hasil selalu sama
- **Efficient**: tidak butuh GPU, selesai dalam detik
- **Grounded in manual labels**: sepenuhnya belajar dari contoh manusia

### Referensi:
- Zhu & Goldberg (2009). Introduction to Semi-Supervised Learning.
- Settles (2012). Active Learning. *Morgan & Claypool.*